# Tedlock: Parse (sentence-level)

OHCO: `part, sent_num, token_num`

Source: existing `tedlock-TOKEN.csv` (spaCy sentence-level); rebuilds DOC/DOCMAP/TOKEN in standard format.

In [ ]:
import pandas as pd
import re

In [ ]:
src_id = 'tedlock'
tok_path = '../../notebooks/tedlock/tedlock-TOKEN.csv'

## Load existing spaCy-sentence TOKEN

In [ ]:
TOK = pd.read_csv(tok_path)
TOK = TOK.rename(columns={'sent': 'sent_num'})
TOK.head()

## Rebuild sentence-level DOC and DOCMAP

In [ ]:
TOK['token_str'] = TOK['token_str'].fillna('').astype(str)
SENT = (TOK.groupby(['part', 'sent_num'])['token_str']
        .apply(' '.join).reset_index().rename(columns={'token_str': 'doc_str'}))
SENT = SENT.reset_index(drop=True); SENT.index.name = 'doc_id'
DOC = SENT[['doc_str']]; DOCMAP = SENT[['part', 'sent_num']]
print(f'{len(DOC):,} sentences')
DOCMAP.head()

## DOC to TOKEN

In [ ]:
TOKEN = DOC.doc_str.str.split(expand=True).stack().to_frame('token_str')
TOKEN.index.names = DOC.index.names + ['token_num']
TOKEN['term_str'] = TOKEN.token_str.str.lower().str.replace(r"[^a-z']", '', regex=True)
TOKEN = TOKEN[TOKEN.term_str != ''].dropna()
TOKEN

## Save

In [ ]:
TOKEN.to_csv(f'{src_id}-TOKEN.csv')
DOC.to_csv(f'{src_id}-DOC.csv')
DOCMAP.to_csv(f'{src_id}-DOCMAP.csv')
print('Saved to notebooks/doc_tables/')